# Post B — What Structure Buys You (Mamba)

This notebook reproduces every figure in Post B of the Latent Dynamics series.  
It expects trained checkpoints and sweep results already on disk — no training happens here.

**What Post B argues:**
1. Mamba achieves O(n) memory and time in sequence length — not by approximating attention, but by using a fundamentally different computational structure: a selective state space model (SSM).
2. The selectivity is learned, not imposed. The model learns *when* to update its hidden state strongly and when to let tokens pass through.
3. Despite the efficiency, Mamba's language modeling performance is competitive with the transformer on enwik8 — especially on positions requiring long-range context.

**The same model, two faces:**  
During training, Mamba can be computed with a parallel associative scan (O(log T) depth).  
During inference, it runs as a sequential RNN — constant memory per step, regardless of context length.

## Cell 1 — Environment setup

Safe to re-run on any machine. If the repo is already cloned and dependencies are installed, nothing changes.

In [ ]:
import subprocess, sys, os

if not os.path.exists('transformer-vs-ssm'):
    subprocess.run(['git', 'clone', 'https://github.com/nvaidyan1/transformer-vs-ssm.git'], check=True)

repo_root = os.path.abspath('transformer-vs-ssm') if os.path.exists('transformer-vs-ssm') else os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)
print(f'Working directory: {os.getcwd()}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pyyaml', 'matplotlib', 'seaborn', 'tqdm'], check=True)
print('Dependencies OK')

## Cell 2 — Config

**This is the only cell you need to edit** if you trained your own models or moved checkpoints.

In [ ]:
import torch

# ── Checkpoint paths — edit these ────────────────────────────────────────────
TRANSFORMER_CKPT = 'checkpoints/transformer/latest.pt'
TCN_CKPT         = 'checkpoints/tcn/latest.pt'
MAMBA_CKPT       = 'checkpoints/mamba/latest.pt'
SWEEP_RESULTS    = 'checkpoints/sweep_results.json'

# ── Output directory for figures ─────────────────────────────────────────────
FIG_DIR = 'figures/post_B'

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {DEVICE}')

# ── Sequence for delta-t visualisation ───────────────────────────────────────
# Pick a sentence where you expect the model to gate selectively:
# content words (nouns, verbs) vs. function words (the, of, in).
DELTA_T_SEQUENCE = 'Napoleon was exiled to the island of Elba in 1814 after his defeat.'

# Optional: annotate specific token positions with vertical lines.
# List of (byte_position, label). Set to [] to skip annotations.
DELTA_T_ANNOTATIONS = [
    (0,  'Napoleon'),
    (27, 'Elba'),
    (35, '1814'),
]

## Cell 3 — How Mamba works

A Mamba block maintains a hidden state **h** of fixed size (d_state per channel) that summarises everything seen so far. At each step t:

```
A_bar = exp(delta_t * A)          # how much to retain from the past
B_bar = delta_t * B_t             # how much to write from the current input
h_t   = A_bar * h_{t-1} + B_bar * x_t
y_t   = C_t · h_t                 # read from state
```

**delta_t** is the key: it is projected from the current token, making it input-dependent (selective).  
- Large delta_t → large A_bar decay → state is heavily updated by this token  
- Small delta_t → state barely changes → token is "skipped over"

This is what distinguishes Mamba from a fixed-coefficient RNN like an LSTM:  
the transition dynamics are a function of the input, learned end-to-end.

In [ ]:
# Sanity check: confirm Mamba parameter counts and config
from src.models import build_model, count_parameters

ckpt = torch.load(MAMBA_CKPT, map_location='cpu')
model_cfg = ckpt['config']['model']
model = build_model('mamba', **model_cfg)
n_params = count_parameters(model)

print('Mamba config:')
for k, v in model_cfg.items():
    print(f'  {k}: {v}')
print(f'  → {n_params:,} trainable parameters')
print(f'\nCheckpoint: step={ckpt["step"]:,}  val_bpc={ckpt["val_bpc"]:.4f}')

## Cell 4 — Verify all checkpoints

In [ ]:
import os

for name, path in [
    ('Transformer',   TRANSFORMER_CKPT),
    ('TCN',           TCN_CKPT),
    ('Mamba',         MAMBA_CKPT),
    ('Sweep results', SWEEP_RESULTS),
]:
    assert os.path.exists(path), f'{name} not found at {path}'
    print(f'{name}: OK  ({path})')

# Confirm all three checkpoints have the expected keys
for arch, path in [('transformer', TRANSFORMER_CKPT), ('tcn', TCN_CKPT), ('mamba', MAMBA_CKPT)]:
    ckpt = torch.load(path, map_location='cpu')
    assert {'model_state', 'val_bpc', 'step', 'config'}.issubset(ckpt.keys())
    print(f'  {arch}: step={ckpt["step"]:,}  val_bpc={ckpt["val_bpc"]:.4f}')

## Cell 5 — Run evaluation (all three models)

Computes overall bpc and long-range bpc for transformer, TCN, and Mamba.  
This cell may take a few minutes depending on hardware.

In [ ]:
from src.data import load_split
from src.dataset import get_dataloader
from src.evaluate import evaluate, evaluate_long_range

val_data = load_split('val')
eval_results = {}

for arch, ckpt_path in [
    ('transformer', TRANSFORMER_CKPT),
    ('tcn',         TCN_CKPT),
    ('mamba',       MAMBA_CKPT),
]:
    ckpt  = torch.load(ckpt_path, map_location='cpu')
    model = build_model(arch, **ckpt['config']['model'])
    model.load_state_dict(ckpt['model_state'])
    model = model.to(DEVICE)

    dl       = get_dataloader('val', seq_len=ckpt['config']['seq_len'],
                              batch_size=8, num_workers=0)
    std_eval = evaluate(model, dl, device=torch.device(DEVICE))
    lr_eval  = evaluate_long_range(model, val_data,
                                   seq_len=ckpt['config']['seq_len'],
                                   device=torch.device(DEVICE))

    eval_results[arch] = {**std_eval, **lr_eval}
    print(f'{arch:>12s}  bpc={std_eval["bpc"]:.4f}  '
          f'long_range_bpc={lr_eval["long_range_bpc"]:.4f}  '
          f'n_lr_positions={lr_eval["n_positions"]:,}')

## Figure 1 — All three models: memory and time vs sequence length

The architectural design space in one picture:
- **Transformer:** O(n²) memory and time — pays for full pairwise attention at every layer
- **TCN:** O(n) in both — fixed convolutional receptive field, no state
- **Mamba:** O(n) in both — recurrent state, but state size is constant regardless of context length

Dashed = O(n²) reference, dotted = O(n) reference, both scaled to the transformer at seq_len=256.

In [ ]:
import json
from IPython.display import Image, display
from src.viz import fig_all_three_sweep

with open(SWEEP_RESULTS) as f:
    sweeps = json.load(f)

out = fig_all_three_sweep(
    transformer_sweep=sweeps['transformer'],
    tcn_sweep=sweeps['tcn'],
    mamba_sweep=sweeps['mamba'],
    output_path=f'{FIG_DIR}/all_three_sweep.png',
)
display(Image(out))

## Figure 2 — Perplexity comparison: overall and long-range

Efficiency only matters if the model still works.  

Solid bars = overall bpc on the full val set.  
Hatched bars = bpc restricted to positions where a byte trigram recurs >512 bytes back (a proxy for long-range dependency positions).

If Mamba's selectivity is doing its job, the gap between its overall and long-range bpc should be smaller than the TCN's — the SSM state can carry relevant context forward across arbitrary distances.

In [ ]:
from src.viz import fig_perplexity_comparison

out = fig_perplexity_comparison(
    results_dict=eval_results,
    output_path=f'{FIG_DIR}/perplexity_comparison.png',
)
display(Image(out))

## Figure 3 — Delta-t across a sequence

This figure makes the selectivity concrete. We run a single sentence through the trained Mamba model and plot the average delta_t at each byte position (averaged across layers and SSM channels).

**Reading the plot:**  
- Peaks at content words (nouns, proper nouns, numbers) indicate the model is gating those tokens strongly into its state.
- Troughs at function words ("the", "of", "in") indicate they are largely passed over.
- This pattern is not hard-coded — it emerges from training on prediction loss alone.

In [ ]:
from src.viz import fig_delta_t

out = fig_delta_t(
    ckpt_path=MAMBA_CKPT,
    sequence=DELTA_T_SEQUENCE,
    output_path=f'{FIG_DIR}/delta_t.png',
    device=DEVICE,
    annotate=DELTA_T_ANNOTATIONS,
)
display(Image(out))

## Figure 4 — Training vs inference schematic

The same Mamba model runs two different computations depending on the context:

- **Training:** the recurrence can be unrolled and computed as a parallel associative scan — a binary tree reduction over the sequence. All outputs are computed simultaneously in O(log T) depth.
- **Inference:** tokens arrive one at a time. The model runs as a true RNN, updating state h_t from h_{t-1} and x_t. Memory cost is O(1) — it does not grow with context length.

This is the core engineering insight: the two faces are mathematically identical (same learned weights, same output), but computationally very different.

In [ ]:
from src.viz import fig_training_inference_schematic

out = fig_training_inference_schematic(
    output_path=f'{FIG_DIR}/training_inference_schematic.png',
)
display(Image(out))

## Summary table — all three models

In [ ]:
print(f'{"arch":>12s}  {"bpc":>8s}  {"ppl":>8s}  {"lr_bpc":>8s}  {"lr_positions":>14s}')
print('-' * 62)
for arch in ['transformer', 'tcn', 'mamba']:
    r = eval_results[arch]
    print(f'{arch:>12s}  {r["bpc"]:>8.4f}  {r["ppl"]:>8.2f}  '
          f'{r["long_range_bpc"]:>8.4f}  {r["n_positions"]:>14,}')

# Confirm all figures were saved
import os
print('\nFigures saved:')
for fname in ['all_three_sweep.png', 'perplexity_comparison.png',
              'delta_t.png', 'training_inference_schematic.png']:
    path = f'{FIG_DIR}/{fname}'
    status = 'OK' if os.path.exists(path) else 'MISSING'
    print(f'  {status}  {path}')